# Video Splitter — YOLOv8 GPU

Same pipeline as `split.ipynb` but using **YOLOv8-Pose on GPU** instead of MediaPipe (CPU only).

YOLOv8-Pose gives us:
- Person detection (count people per frame)
- Pose keypoints (track body center for action jump detection)
- All on GPU → much faster

In [1]:
import numpy as np
import json
import os
import torch
from ultralytics import YOLO
from moviepy import VideoFileClip

print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA: True
GPU: NVIDIA GeForce RTX 5090


In [2]:
# ── Config ──────────────────────────────────────────
VIDEO_PATH = "video.mp4"
OUTPUT_DIR = "subclips_yolo"
JSON_PATH = "chunks_yolo.json"

SAMPLE_EVERY = 1         # analyse every Nth frame
MIN_CHUNK_FRAMES = 50     # minimum frames to keep a chunk
JUMP_THRESHOLD = 0.15     # normalised landmark displacement for action jump
NO_HUMAN_GAP = 25         # consecutive no-human frames to end a segment
CENTER_MARGIN = 0.15      # person must be within 0.5 ± margin horizontally
PERSON_CONF = 0.5         # minimum confidence for person detection

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load YOLOv8-Pose model (auto-downloads on first run)
model = YOLO("yolo11n-pose.pt")  # nano model, fast
model.to("cuda")
print(f"Model loaded on {next(model.model.parameters()).device}")

Model loaded on cuda:0


## Step 1 — Scan video with YOLO-Pose on GPU

In [3]:
video_clip = VideoFileClip(VIDEO_PATH)
fps = video_clip.fps
total_frames = int(video_clip.duration * fps)
w, h = video_clip.size
print(f"Video: {total_frames} frames, {fps} FPS, ~{video_clip.duration:.0f}s, {w}x{h}")

# (frame_idx, valid_single_person, normalized_center)
frame_data = []
multi_person_skipped = 0
off_center_skipped = 0

KP_NOSE, KP_LHIP, KP_RHIP = 0, 11, 12
BATCH_SIZE = 32  # frames per GPU batch

# Collect sampled frames into batches
batch_frames = []
batch_indices = []

def process_batch(batch_frames, batch_indices):
    """Run YOLO on a batch of frames and return results."""
    global multi_person_skipped, off_center_skipped
    results_list = model.predict(batch_frames, classes=[0], conf=PERSON_CONF, verbose=False)

    for res, fidx in zip(results_list, batch_indices):
        n_persons = len(res.boxes)

        if n_persons == 0:
            frame_data.append((fidx, False, None))
        elif n_persons > 1:
            frame_data.append((fidx, False, None))
            multi_person_skipped += 1
        else:
            kpts = res.keypoints.xy[0].cpu().numpy()
            nose, lhip, rhip = kpts[KP_NOSE], kpts[KP_LHIP], kpts[KP_RHIP]

            if nose[0] == 0 and lhip[0] == 0 and rhip[0] == 0:
                frame_data.append((fidx, False, None))
            else:
                valid = [p for p in [nose, lhip, rhip] if p[0] > 0]
                cx = np.mean([p[0] for p in valid]) / w
                cy = np.mean([p[1] for p in valid]) / h

                if abs(cx - 0.5) > CENTER_MARGIN:
                    frame_data.append((fidx, False, None))
                    off_center_skipped += 1
                else:
                    frame_data.append((fidx, True, (cx, cy)))

for frame_idx, frame in enumerate(video_clip.iter_frames(dtype="uint8")):
    if frame_idx % SAMPLE_EVERY != 0:
        continue

    batch_frames.append(frame)
    batch_indices.append(frame_idx)

    if len(batch_frames) >= BATCH_SIZE:
        process_batch(batch_frames, batch_indices)
        batch_frames, batch_indices = [], []

    if frame_idx % 5000 == 0 and frame_idx > 0:
        print(f"  scanned {frame_idx}/{total_frames} frames...")

# Process remaining frames
if batch_frames:
    process_batch(batch_frames, batch_indices)

video_clip.close()
human_count = sum(1 for _, h, _ in frame_data if h)
no_human = len(frame_data) - human_count - multi_person_skipped - off_center_skipped
print(f"\nScan complete. {len(frame_data)} sampled frames:")
print(f"  - {human_count} with single centered human")
print(f"  - {multi_person_skipped} skipped (multiple people)")
print(f"  - {off_center_skipped} skipped (person not centered)")
print(f"  - {no_human} no human detected")

Video: 94330 frames, 25.0 FPS, ~3773s, 1920x1080
  scanned 5000/94330 frames...
  scanned 10000/94330 frames...
  scanned 15000/94330 frames...
  scanned 20000/94330 frames...
  scanned 25000/94330 frames...
  scanned 30000/94330 frames...
  scanned 35000/94330 frames...
  scanned 40000/94330 frames...
  scanned 45000/94330 frames...
  scanned 50000/94330 frames...
  scanned 55000/94330 frames...
  scanned 60000/94330 frames...
  scanned 65000/94330 frames...
  scanned 70000/94330 frames...
  scanned 75000/94330 frames...
  scanned 80000/94330 frames...
  scanned 85000/94330 frames...
  scanned 90000/94330 frames...

Scan complete. 94330 sampled frames:
  - 90161 with single centered human
  - 49 skipped (multiple people)
  - 114 skipped (person not centered)
  - 4006 no human detected


/home/dipcik/miniconda3/envs/tracker/lib/python3.10/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file video.mp4, 6220800 bytes wanted but 0 bytes read at frame index 94329 (out of a total 94330 frames), at time 3773.16/3773.22 sec. Using the last valid frame instead.
  warnings.warn(


## Step 2 — Find human-present segments & detect action jumps

In [4]:
def find_chunks(frame_data, no_human_gap, jump_threshold, min_chunk_frames, sample_every):
    chunks = []
    current_start = None
    prev_center = None
    gap_count = 0

    for fidx, has_human, center in frame_data:
        if has_human:
            if current_start is None:
                current_start = fidx
                prev_center = center
                gap_count = 0
                continue

            gap_count = 0

            if prev_center is not None:
                dist = np.sqrt((center[0] - prev_center[0])**2 +
                               (center[1] - prev_center[1])**2)
                if dist > jump_threshold:
                    chunk_end = fidx - 1
                    if (chunk_end - current_start) >= min_chunk_frames:
                        chunks.append((current_start, chunk_end))
                    current_start = fidx

            prev_center = center

        else:
            if current_start is not None:
                gap_count += sample_every
                if gap_count >= no_human_gap:
                    chunk_end = fidx - gap_count
                    if (chunk_end - current_start) >= min_chunk_frames:
                        chunks.append((current_start, chunk_end))
                    current_start = None
                    prev_center = None
                    gap_count = 0

    if current_start is not None:
        last_frame = frame_data[-1][0]
        if (last_frame - current_start) >= min_chunk_frames:
            chunks.append((current_start, last_frame))

    return chunks


chunks = find_chunks(frame_data, NO_HUMAN_GAP, JUMP_THRESHOLD, MIN_CHUNK_FRAMES, SAMPLE_EVERY)
print(f"Found {len(chunks)} chunks:")
for i, (s, e) in enumerate(chunks):
    dur = (e - s) / fps
    print(f"  chunk {i:03d}: frames {s} - {e}  ({dur:.1f}s)")

Found 44 chunks:
  chunk 000: frames 0 - 51  (2.0s)
  chunk 001: frames 52 - 115  (2.5s)
  chunk 002: frames 158 - 224  (2.6s)
  chunk 003: frames 225 - 309  (3.4s)
  chunk 004: frames 460 - 514  (2.2s)
  chunk 005: frames 515 - 573  (2.3s)
  chunk 006: frames 685 - 736  (2.0s)
  chunk 007: frames 781 - 841  (2.4s)
  chunk 008: frames 926 - 985  (2.4s)
  chunk 009: frames 1142 - 1193  (2.0s)
  chunk 010: frames 1194 - 1258  (2.6s)
  chunk 011: frames 1347 - 1432  (3.4s)
  chunk 012: frames 1584 - 5533  (158.0s)
  chunk 013: frames 5645 - 5774  (5.2s)
  chunk 014: frames 5900 - 14511  (344.4s)
  chunk 015: frames 14637 - 17181  (101.8s)
  chunk 016: frames 17307 - 18780  (58.9s)
  chunk 017: frames 18906 - 20607  (68.0s)
  chunk 018: frames 20733 - 22183  (58.0s)
  chunk 019: frames 22309 - 22585  (11.0s)
  chunk 020: frames 22711 - 23595  (35.4s)
  chunk 021: frames 23701 - 37673  (558.9s)
  chunk 022: frames 37776 - 45356  (303.2s)
  chunk 023: frames 45462 - 46876  (56.6s)
  chunk 02

## Step 3 — Export subclips with audio & save JSON

In [5]:
chunks_meta = []
for i, (start_frame, end_frame) in enumerate(chunks):
    chunks_meta.append({
        "chunk_id": i,
        "start_frame": int(start_frame),
        "end_frame": int(end_frame),
        "start_time_s": round(start_frame / fps, 3),
        "end_time_s": round(end_frame / fps, 3),
        "duration_s": round((end_frame - start_frame) / fps, 3),
        "filename": f"chunk_{i:03d}.mp4"
    })

with open(JSON_PATH, "w") as f:
    json.dump(chunks_meta, f, indent=2)

print(f"Saved {JSON_PATH} with {len(chunks_meta)} entries.")

Saved chunks_yolo.json with 44 entries.


In [6]:
video = VideoFileClip(VIDEO_PATH)

for meta in chunks_meta:
    t_start = meta["start_time_s"]
    t_end = meta["end_time_s"]
    out_path = os.path.join(OUTPUT_DIR, meta["filename"])

    subclip = video.subclipped(t_start, t_end)
    subclip.write_videofile(
        out_path,
        codec="libx264",
        audio_codec="aac",
        logger="bar",
    )
    print(f"  >> {meta['filename']} ({meta['duration_s']:.1f}s)")

video.close()
print(f"\nDone! {len(chunks_meta)} subclips saved to {OUTPUT_DIR}/")

MoviePy - Building video subclips_yolo/chunk_000.mp4.
MoviePy - Writing audio in chunk_000TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
MoviePy - Writing video subclips_yolo/chunk_000.mp4



MoviePy - Done !
MoviePy - video ready subclips_yolo/chunk_000.mp4
  >> chunk_000.mp4 (2.0s)
MoviePy - Building video subclips_yolo/chunk_001.mp4.
MoviePy - Writing audio in chunk_001TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
MoviePy - Writing video subclips_yolo/chunk_001.mp4



MoviePy - Done !
MoviePy - video ready subclips_yolo/chunk_001.mp4
  >> chunk_001.mp4 (2.5s)
MoviePy - Building video subclips_yolo/chunk_002.mp4.
MoviePy - Writing audio in chunk_002TEMP_MPY_wvf_snd.mp4


...

MoviePy - Done !
MoviePy - video ready subclips_yolo/chunk_041.mp4
  >> chunk_041.mp4 (55.1s)
MoviePy - Building video subclips_yolo/chunk_042.mp4.
MoviePy - Writing audio in chunk_042TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
MoviePy - Writing video subclips_yolo/chunk_042.mp4



MoviePy - Done !
MoviePy - video ready subclips_yolo/chunk_042.mp4
  >> chunk_042.mp4 (75.0s)
MoviePy - Building video subclips_yolo/chunk_043.mp4.
MoviePy - Writing audio in chunk_043TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
MoviePy - Writing video subclips_yolo/chunk_043.mp4



MoviePy - Done !
MoviePy - video ready subclips_yolo/chunk_043.mp4
  >> chunk_043.mp4 (186.4s)

Done! 44 subclips saved to subclips_yolo/


## Check — Verify chunks frame by frame

In [7]:
import glob

chunk_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "chunk_*.mp4")))
print(f"Checking {len(chunk_files)} chunks for missing person detections...\n")

CHECK_EVERY = 1
CHECK_BATCH = 32
flagged = []

for chunk_path in chunk_files:
    fname = os.path.basename(chunk_path)
    clip = VideoFileClip(chunk_path)
    n_frames = int(clip.duration * clip.fps)
    no_person_frames = []

    batch_f, batch_i = [], []
    for fidx, frame in enumerate(clip.iter_frames(dtype="uint8")):
        if fidx % CHECK_EVERY != 0:
            continue
        batch_f.append(frame)
        batch_i.append(fidx)

        if len(batch_f) >= CHECK_BATCH:
            results_list = model.predict(batch_f, classes=[0], conf=PERSON_CONF, verbose=False)
            for res, idx in zip(results_list, batch_i):
                if len(res.boxes) == 0:
                    no_person_frames.append(idx)
            batch_f, batch_i = [], []

    if batch_f:
        results_list = model.predict(batch_f, classes=[0], conf=PERSON_CONF, verbose=False)
        for res, idx in zip(results_list, batch_i):
            if len(res.boxes) == 0:
                no_person_frames.append(idx)

    clip.close()
    checked = n_frames // CHECK_EVERY
    ratio = len(no_person_frames) / max(checked, 1)

    if no_person_frames:
        flagged.append({
            "filename": fname,
            "total_frames": n_frames,
            "checked_frames": checked,
            "no_person_frames": len(no_person_frames),
            "ratio": round(ratio, 3),
            "sample_missing": no_person_frames[:10],
        })
        print(f"  {fname}: {len(no_person_frames)}/{checked} frames no person ({ratio*100:.1f}%)")
    else:
        print(f"  {fname}: OK")

print(f"\n{'='*50}")
print(f"Flagged: {len(flagged)}/{len(chunk_files)} chunks have frames without person")
for f in flagged:
    print(f"  {f['filename']}: {f['no_person_frames']} missing in {f['checked_frames']} checked ({f['ratio']*100:.1f}%)")

Checking 44 chunks for missing person detections...

  chunk_000.mp4: OK


/home/dipcik/miniconda3/envs/tracker/lib/python3.10/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file subclips_yolo/chunk_001.mp4, 6220800 bytes wanted but 0 bytes read at frame index 62 (out of a total 63 frames), at time 2.48/2.52 sec. Using the last valid frame instead.
  warnings.warn(


  chunk_001.mp4: OK
  chunk_002.mp4: OK


/home/dipcik/miniconda3/envs/tracker/lib/python3.10/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file subclips_yolo/chunk_003.mp4, 6220800 bytes wanted but 0 bytes read at frame index 83 (out of a total 84 frames), at time 3.32/3.36 sec. Using the last valid frame instead.
  warnings.warn(


  chunk_003.mp4: OK
  chunk_004.mp4: OK
  chunk_005.mp4: OK
  chunk_006.mp4: OK
  chunk_007.mp4: OK


...

  chunk_032.mp4: 19/944 frames no person (2.0%)
  chunk_033.mp4: 21/3700 frames no person (0.6%)
  chunk_034.mp4: 22/683 frames no person (3.2%)


/home/dipcik/miniconda3/envs/tracker/lib/python3.10/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file subclips_yolo/chunk_035.mp4, 6220800 bytes wanted but 0 bytes read at frame index 2881 (out of a total 2882 frames), at time 115.24/115.28 sec. Using the last valid frame instead.
  warnings.warn(


  chunk_035.mp4: 19/2882 frames no person (0.7%)
  chunk_036.mp4: 19/860 frames no person (2.2%)
  chunk_037.mp4: 19/1297 frames no person (1.5%)
  chunk_038.mp4: 19/1994 frames no person (1.0%)
  chunk_039.mp4: 21/1692 frames no person (1.2%)
  chunk_040.mp4: 18/1680 frames no person (1.1%)


/home/dipcik/miniconda3/envs/tracker/lib/python3.10/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file subclips_yolo/chunk_041.mp4, 6220800 bytes wanted but 0 bytes read at frame index 1376 (out of a total 1377 frames), at time 55.04/55.08 sec. Using the last valid frame instead.
  warnings.warn(


  chunk_041.mp4: 19/1377 frames no person (1.4%)
  chunk_042.mp4: 18/1875 frames no person (1.0%)
  chunk_043.mp4: OK

Flagged: 23/44 chunks have frames without person
  chunk_020.mp4: 19 missing in 884 checked (2.1%)
  chunk_021.mp4: 21 missing in 13972 checked (0.2%)
  chunk_022.mp4: 19 missing in 7580 checked (0.3%)
  chunk_023.mp4: 19 missing in 1414 checked (1.3%)
  chunk_024.mp4: 19 missing in 6168 checked (0.3%)
  chunk_025.mp4: 19 missing in 2947 checked (0.6%)
  chunk_026.mp4: 19 missing in 1355 checked (1.4%)
  chunk_027.mp4: 19 missing in 1332 checked (1.4%)
  chunk_028.mp4: 19 missing in 2809 checked (0.7%)
  chunk_029.mp4: 19 missing in 2283 checked (0.8%)
  chunk_030.mp4: 19 missing in 1902 checked (1.0%)
  chunk_031.mp4: 19 missing in 2786 checked (0.7%)
  chunk_032.mp4: 19 missing in 944 checked (2.0%)
  chunk_033.mp4: 21 missing in 3700 checked (0.6%)
  chunk_034.mp4: 22 missing in 683 checked (3.2%)
  chunk_035.mp4: 19 missing in 2882 checked (0.7%)
  chunk_036.mp4: 1

## Trim — Remove last N no-person frames from flagged chunks

In [10]:
import subprocess

trimmed_count = 0
for f in flagged:
    fname = f["filename"]
    n_missing = f["no_person_frames"]
    chunk_path = os.path.join(OUTPUT_DIR, fname)

    # Get actual duration via ffprobe
    probe = subprocess.run(
        ["ffprobe", "-v", "quiet", "-show_entries", "format=duration",
         "-of", "csv=p=0", chunk_path],
        capture_output=True, text=True
    )
    duration = float(probe.stdout.strip())
    clip_fps = fps  # same as source
    trim_seconds = n_missing / clip_fps
    new_end = duration - trim_seconds

    if new_end <= 0.5:
        print(f"  SKIP {fname}: would trim entire video")
        continue

    # Use ffmpeg stream copy — no re-encode, instant
    tmp_path = chunk_path.replace(".mp4", "_trimmed.mp4")
    subprocess.run([
        "ffmpeg", "-y", "-i", chunk_path,
        "-t", str(new_end),
        "-c", "copy",  # no re-encode
        tmp_path
    ], capture_output=True)

    os.replace(tmp_path, chunk_path)
    trimmed_count += 1
    print(f"  {fname}: trimmed {n_missing} frames ({trim_seconds:.2f}s) -> {new_end:.2f}s")

# Update chunks_yolo.json
with open(JSON_PATH, "r") as jf:
    chunks_meta = json.load(jf)

for meta in chunks_meta:
    for f in flagged:
        if meta["filename"] == f["filename"]:
            n_missing = f["no_person_frames"]
            trim_s = n_missing / fps
            meta["end_time_s"] = round(meta["end_time_s"] - trim_s, 3)
            meta["end_frame"] = int(meta["end_frame"] - n_missing)
            meta["duration_s"] = round(meta["end_time_s"] - meta["start_time_s"], 3)
            break

with open(JSON_PATH, "w") as jf:
    json.dump(chunks_meta, jf, indent=2)

print(f"\nDone! Trimmed {trimmed_count}/{len(flagged)} chunks. Updated {JSON_PATH}.")

  chunk_020.mp4: trimmed 19 frames (0.76s) -> 33.08s
  chunk_021.mp4: trimmed 21 frames (0.84s) -> 558.04s
  chunk_022.mp4: trimmed 19 frames (0.76s) -> 302.44s
  chunk_023.mp4: trimmed 19 frames (0.76s) -> 55.80s
  chunk_024.mp4: trimmed 19 frames (0.76s) -> 245.96s
  chunk_025.mp4: trimmed 19 frames (0.76s) -> 117.12s
  chunk_026.mp4: trimmed 19 frames (0.76s) -> 53.44s
  chunk_027.mp4: trimmed 19 frames (0.76s) -> 52.52s
  chunk_028.mp4: trimmed 19 frames (0.76s) -> 111.60s
  chunk_029.mp4: trimmed 19 frames (0.76s) -> 90.56s
  chunk_030.mp4: trimmed 19 frames (0.76s) -> 75.32s
  chunk_031.mp4: trimmed 19 frames (0.76s) -> 110.68s
  chunk_032.mp4: trimmed 19 frames (0.76s) -> 37.00s
  chunk_033.mp4: trimmed 21 frames (0.84s) -> 147.16s
  chunk_034.mp4: trimmed 22 frames (0.88s) -> 26.44s
  chunk_035.mp4: trimmed 19 frames (0.76s) -> 114.52s
  chunk_036.mp4: trimmed 19 frames (0.76s) -> 33.64s
  chunk_037.mp4: trimmed 19 frames (0.76s) -> 51.12s
  chunk_038.mp4: trimmed 19 frames (0.

## Check — Verify again chunks frame by frame after trim

In [11]:
chunk_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "chunk_*.mp4")))
print(f"Re-checking {len(chunk_files)} chunks after trim...\n")

CHECK_BATCH = 32
flagged_after = []

for chunk_path in chunk_files:
    fname = os.path.basename(chunk_path)
    clip = VideoFileClip(chunk_path)
    n_frames = int(clip.duration * clip.fps)
    no_person_frames = []

    batch_f, batch_i = [], []
    for fidx, frame in enumerate(clip.iter_frames(dtype="uint8")):
        batch_f.append(frame)
        batch_i.append(fidx)

        if len(batch_f) >= CHECK_BATCH:
            results_list = model.predict(batch_f, classes=[0], conf=PERSON_CONF, verbose=False)
            for res, idx in zip(results_list, batch_i):
                if len(res.boxes) == 0:
                    no_person_frames.append(idx)
            batch_f, batch_i = [], []

    if batch_f:
        results_list = model.predict(batch_f, classes=[0], conf=PERSON_CONF, verbose=False)
        for res, idx in zip(results_list, batch_i):
            if len(res.boxes) == 0:
                no_person_frames.append(idx)

    clip.close()
    ratio = len(no_person_frames) / max(n_frames, 1)

    if no_person_frames:
        flagged_after.append({
            "filename": fname,
            "total_frames": n_frames,
            "no_person_frames": len(no_person_frames),
            "ratio": round(ratio, 3),
        })
        print(f"  {fname}: {len(no_person_frames)}/{n_frames} frames no person ({ratio*100:.1f}%)")
    else:
        print(f"  {fname}: OK")

print(f"\n{'='*50}")
if flagged_after:
    print(f"Still flagged: {len(flagged_after)}/{len(chunk_files)} chunks")
    for f in flagged_after:
        print(f"  {f['filename']}: {f['no_person_frames']}/{f['total_frames']} ({f['ratio']*100:.1f}%)")
else:
    print("All chunks clean! Every frame has a person detected.")

Re-checking 44 chunks after trim...

  chunk_000.mp4: OK


/home/dipcik/miniconda3/envs/tracker/lib/python3.10/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file subclips_yolo/chunk_001.mp4, 6220800 bytes wanted but 0 bytes read at frame index 62 (out of a total 63 frames), at time 2.48/2.52 sec. Using the last valid frame instead.
  warnings.warn(


  chunk_001.mp4: OK
  chunk_002.mp4: OK


/home/dipcik/miniconda3/envs/tracker/lib/python3.10/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file subclips_yolo/chunk_003.mp4, 6220800 bytes wanted but 0 bytes read at frame index 83 (out of a total 84 frames), at time 3.32/3.36 sec. Using the last valid frame instead.
  warnings.warn(


  chunk_003.mp4: OK
  chunk_004.mp4: OK
  chunk_005.mp4: OK
  chunk_006.mp4: OK
  chunk_007.mp4: OK


...

  chunk_012.mp4: OK


/home/dipcik/miniconda3/envs/tracker/lib/python3.10/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file subclips_yolo/chunk_013.mp4, 6220800 bytes wanted but 0 bytes read at frame index 128 (out of a total 129 frames), at time 5.12/5.16 sec. Using the last valid frame instead.
  warnings.warn(


  chunk_013.mp4: OK
  chunk_014.mp4: OK
  chunk_015.mp4: OK
  chunk_016.mp4: OK
  chunk_017.mp4: OK
  chunk_018.mp4: OK


/home/dipcik/miniconda3/envs/tracker/lib/python3.10/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file subclips_yolo/chunk_019.mp4, 6220800 bytes wanted but 0 bytes read at frame index 275 (out of a total 276 frames), at time 11.00/11.04 sec. Using the last valid frame instead.
  warnings.warn(


  chunk_019.mp4: OK
  chunk_020.mp4: OK
  chunk_021.mp4: 3/13953 frames no person (0.0%)
  chunk_022.mp4: 2/7563 frames no person (0.0%)
  chunk_023.mp4: 2/1397 frames no person (0.1%)
  chunk_024.mp4: 2/6151 frames no person (0.0%)
  chunk_025.mp4: 2/2930 frames no person (0.1%)
  chunk_026.mp4: 5/1338 frames no person (0.4%)
  chunk_027.mp4: 2/1315 frames no person (0.2%)
  chunk_028.mp4: 2/2792 frames no person (0.1%)
  chunk_029.mp4: 2/2266 frames no person (0.1%)
  chunk_030.mp4: 2/1885 frames no person (0.1%)
  chunk_031.mp4: 2/2769 frames no person (0.1%)
  chunk_032.mp4: 2/927 frames no person (0.2%)
  chunk_033.mp4: 4/3681 frames no person (0.1%)
  chunk_034.mp4: 3/663 frames no person (0.5%)
  chunk_035.mp4: 2/2865 frames no person (0.1%)
  chunk_036.mp4: 2/843 frames no person (0.2%)
  chunk_037.mp4: 2/1280 frames no person (0.2%)
  chunk_038.mp4: 2/1977 frames no person (0.1%)
  chunk_039.mp4: 3/1673 frames no person (0.2%)
  chunk_040.mp4: 2/1664 frames no person (0.1%)
  

In [12]:
# Trim last 10 frames from all flagged_after chunks via ffmpeg stream copy
import subprocess

TRIM_FRAMES = 10
trimmed2 = 0

for f in flagged_after:
    fname = f["filename"]
    chunk_path = os.path.join(OUTPUT_DIR, fname)

    probe = subprocess.run(
        ["ffprobe", "-v", "quiet", "-show_entries", "format=duration",
         "-of", "csv=p=0", chunk_path],
        capture_output=True, text=True
    )
    duration = float(probe.stdout.strip())
    trim_s = TRIM_FRAMES / fps
    new_end = duration - trim_s

    if new_end <= 0.5:
        print(f"  SKIP {fname}")
        continue

    tmp_path = chunk_path.replace(".mp4", "_trimmed.mp4")
    subprocess.run([
        "ffmpeg", "-y", "-i", chunk_path,
        "-t", str(new_end),
        "-c", "copy",
        tmp_path
    ], capture_output=True)

    os.replace(tmp_path, chunk_path)
    trimmed2 += 1
    print(f"  {fname}: trimmed last {TRIM_FRAMES} frames ({trim_s:.2f}s) -> {new_end:.2f}s")

# Update chunks_yolo.json
with open(JSON_PATH, "r") as jf:
    chunks_meta = json.load(jf)

flagged_names = {f["filename"] for f in flagged_after}
for meta in chunks_meta:
    if meta["filename"] in flagged_names:
        trim_s = TRIM_FRAMES / fps
        meta["end_time_s"] = round(meta["end_time_s"] - trim_s, 3)
        meta["end_frame"] = int(meta["end_frame"] - TRIM_FRAMES)
        meta["duration_s"] = round(meta["end_time_s"] - meta["start_time_s"], 3)

with open(JSON_PATH, "w") as jf:
    json.dump(chunks_meta, jf, indent=2)

print(f"\nDone! Trimmed {trimmed2}/{len(flagged_after)} chunks. Updated {JSON_PATH}.")

  chunk_021.mp4: trimmed last 10 frames (0.40s) -> 557.72s
  chunk_022.mp4: trimmed last 10 frames (0.40s) -> 302.12s
  chunk_023.mp4: trimmed last 10 frames (0.40s) -> 55.48s
  chunk_024.mp4: trimmed last 10 frames (0.40s) -> 245.64s
  chunk_025.mp4: trimmed last 10 frames (0.40s) -> 116.80s
  chunk_026.mp4: trimmed last 10 frames (0.40s) -> 53.12s
  chunk_027.mp4: trimmed last 10 frames (0.40s) -> 52.20s
  chunk_028.mp4: trimmed last 10 frames (0.40s) -> 111.28s
  chunk_029.mp4: trimmed last 10 frames (0.40s) -> 90.24s
  chunk_030.mp4: trimmed last 10 frames (0.40s) -> 75.00s
  chunk_031.mp4: trimmed last 10 frames (0.40s) -> 110.36s
  chunk_032.mp4: trimmed last 10 frames (0.40s) -> 36.68s
  chunk_033.mp4: trimmed last 10 frames (0.40s) -> 146.84s
  chunk_034.mp4: trimmed last 10 frames (0.40s) -> 26.12s
  chunk_035.mp4: trimmed last 10 frames (0.40s) -> 114.20s
  chunk_036.mp4: trimmed last 10 frames (0.40s) -> 33.32s
  chunk_037.mp4: trimmed last 10 frames (0.40s) -> 50.80s
  chun

In [ ]:
# Calculate remaining video length from actual files
import subprocess

chunk_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "chunk_*.mp4")))
total_duration = 0

for chunk_path in chunk_files:
    probe = subprocess.run(
        ["ffprobe", "-v", "quiet", "-show_entries", "format=duration",
         "-of", "csv=p=0", chunk_path],
        capture_output=True, text=True
    )
    total_duration += float(probe.stdout.strip())

orig_duration = total_frames / fps
removed = orig_duration - total_duration

print(f"Chunks: {len(chunk_files)}")
print(f"Total remaining: {total_duration:.1f}s ({total_duration/60:.1f} min)")
print(f"Original:        {orig_duration:.1f}s ({orig_duration/60:.1f} min)")
print(f"Removed:         {removed:.1f}s ({removed/60:.1f} min) = {removed/orig_duration*100:.1f}%")

Chunks: 32
Total remaining: 3543.9s (59.1 min)
Original:        3773.2s (62.9 min)
Removed:         229.3s (3.8 min) = 6.1%


: 

## Summary